# PC情報抽出ツール
キッティングログからシリアル番号・MACアドレス・BitLocker回復キーを抽出し、Excelに出力します。

## 使い方
1. 下記フォルダ構造のZIPファイルをアップロードしてください
```
PCログ.zip
  ├── LA220170/
  │   ├── LA220170_kitting_log.txt
  │   ├── LA220170-C.txt
  │   └── LA220170-D.txt
  ├── LA220171/
  │   ├── ...
  └── ...
```
2. セルを上から順に実行してください
3. 最後に `PC情報一覧.xlsx` がダウンロードされます

In [4]:
# --- セル1: ZIPファイルのアップロードと展開 ---
from google.colab import files
import zipfile, os, shutil

WORK_DIR = '/content/pc_logs'
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR)

print('ZIPファイルをアップロードしてください')
uploaded = files.upload()

for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall(WORK_DIR)
        print(f'展開完了: {fname}')
    else:
        print(f'スキップ（ZIP以外）: {fname}')

# フォルダ一覧を表示
folders = sorted([d for d in os.listdir(WORK_DIR)
                  if os.path.isdir(os.path.join(WORK_DIR, d)) and not d.startswith('__')])

# ZIPの中にさらに1段フォルダがある場合の対応
if len(folders) == 1:
    inner = os.path.join(WORK_DIR, folders[0])
    sub = [d for d in os.listdir(inner) if os.path.isdir(os.path.join(inner, d))]
    if len(sub) > 1:
        WORK_DIR = inner
        folders = sorted(sub)

print(f'\n検出されたPCフォルダ数: {len(folders)}')
for f in folders[:10]:
    print(f'  {f}')
if len(folders) > 10:
    print(f'  ... 他 {len(folders)-10} フォルダ')

ZIPファイルをアップロードしてください


Saving HB_log.zip to HB_log.zip
展開完了: HB_log.zip

検出されたPCフォルダ数: 321
  LA220001
  LA220002
  LA220003
  LA220004
  LA220005
  LA220006
  LA220007
  LA220008
  LA220009
  LA220010
  ... 他 311 フォルダ


In [5]:
# --- セル2: ログ解析・Excel出力 ---
import re
import pandas as pd
from google.colab import files as colab_files

def read_file(filepath):
    for enc in ['cp932', 'utf-8-sig', 'utf-8']:
        try:
            with open(filepath, 'r', encoding=enc) as f:
                return f.read()
        except (UnicodeDecodeError, FileNotFoundError):
            continue
    return None

def extract_serial(text):
    m = re.search(r'(?:SerialNumber|serialnumber)\s*\n([A-Za-z0-9]+)', text, re.IGNORECASE)
    if m:
        return m.group(1).strip()
    m = re.search(r'シリアル.*?\n([A-Za-z0-9]+)', text)
    if m:
        return m.group(1).strip()
    return ''

def extract_mac_entries(text):
    entries = []
    mac_section = re.split(r'====.*?MAC.*?====', text)
    if len(mac_section) < 2:
        return entries
    mac_text = mac_section[1].split('====')[0]
    blocks = re.split(r'接続名\s*[:：]\s*', mac_text)
    for block in blocks[1:]:
        lines = block.strip().split('\n')
        conn_name = lines[0].strip() if lines else ''
        adapter = ''
        mac = ''
        for line in lines:
            if re.search(r'(?:アダプター|アダプタ)', line):
                m = re.search(r'[:：]\s*(.+)', line)
                if m:
                    adapter = m.group(1).strip()
            m = re.search(r'([0-9A-Fa-f]{2}[-:][0-9A-Fa-f]{2}[-:][0-9A-Fa-f]{2}[-:][0-9A-Fa-f]{2}[-:][0-9A-Fa-f]{2}[-:][0-9A-Fa-f]{2})', line)
            if m:
                mac = m.group(1)
        if mac:
            entries.append({'接続名': conn_name, 'アダプタ': adapter, 'MACアドレス': mac})
    return entries

def extract_recovery_key(text):
    vol, key_id, password = '', '', ''
    m = re.search(r'([A-Z])\s*:\s*\[', text)
    if m:
        vol = m.group(1)
    m = re.search(r'ID\s*[:：]\s*\{?([A-Fa-f0-9-]+)\}?', text)
    if m:
        key_id = m.group(1)
    m = re.search(r'(\d{6}-\d{6}-\d{6}-\d{6}-\d{6}-\d{6}-\d{6}-\d{6})', text)
    if m:
        password = m.group(1)
    return vol, key_id, password

# ===== メイン処理 =====
rows = []
errors = []

for folder_name in folders:
    folder_path = os.path.join(WORK_DIR, folder_name)
    if not os.path.isdir(folder_path):
        continue

    pc_name = folder_name
    serial = ''
    mac_entries = []
    recovery_keys = []

    for fname in sorted(os.listdir(folder_path)):
        fpath = os.path.join(folder_path, fname)
        if not os.path.isfile(fpath):
            continue
        text = read_file(fpath)
        if text is None:
            errors.append(f'読み取り失敗: {fpath}')
            continue

        if 'kitting_log' in fname.lower() or ('log' in fname.lower() and not re.match(r'.*-[A-Z]\.txt$', fname)):
            s = extract_serial(text)
            if s:
                serial = s
            mac_entries = extract_mac_entries(text)

        if re.match(r'.*-[A-Z]\.txt$', fname, re.IGNORECASE):
            vol, key_id, password = extract_recovery_key(text)
            if password:
                recovery_keys.append((vol, key_id, password))

    recovery_keys.sort(key=lambda x: x[0])

    # --- 1PC = 1行 ---
    row = {'PC名': pc_name, 'シリアル番号': serial}

    for i, me in enumerate(mac_entries, 1):
        row[f'NIC{i}_接続名'] = me['接続名']
        row[f'NIC{i}_アダプタ'] = me['アダプタ']
        row[f'NIC{i}_MAC'] = me['MACアドレス']

    for rk in recovery_keys:
        vol = rk[0] or '?'
        row[f'{vol}ドライブ_キーID'] = rk[1]
        row[f'{vol}ドライブ_回復キー'] = rk[2]

    rows.append(row)

df = pd.DataFrame(rows)

# 列順を整理
base_cols = ['PC名', 'シリアル番号']
nic_cols = sorted([c for c in df.columns if c.startswith('NIC')])
key_cols = sorted([c for c in df.columns if 'ドライブ' in c])
df = df[base_cols + nic_cols + key_cols].fillna('')

print(f'処理完了: {len(folders)} 台のPC（{len(df.columns)} 列）')
if errors:
    print(f'\nエラー ({len(errors)}件):')
    for e in errors:
        print(f'  {e}')

df

処理完了: 321 台のPC（12 列）


,PC名,シリアル番号,NIC1_MAC,NIC1_アダプタ,NIC1_接続名,NIC2_MAC,NIC2_アダプタ,NIC2_接続名,Cドライブ_キーID,Cドライブ_回復キー,Dドライブ_キーID,Dドライブ_回復キー
0,LA220001,R2901233,E4-46-B0-18-33-73,Intel(R) Ethernet Connection (16) I219-LM,イーサネット,3C-21-9C-CA-C9-CF,Intel(R) Wi-Fi 6E AX211 160MHz,Wi-Fi,4FE9E34F-5345-4B04-9336-7093AB742786,612634-091938-274879-108108-478896-171281-2718...,8A019A57-807C-44AD-BAE2-9EABE4C10096,198286-699897-142791-670747-693154-298133-4819...
1,LA220002,R2901414,E4-46-B0-18-41-8B,Intel(R) Ethernet Connection (16) I219-LM,イーサネット,3C-21-9C-C9-5A-63,Intel(R) Wi-Fi 6E AX211 160MHz,Wi-Fi,3C7B854E-F771-4ED9-83DE-16B17DE89B0D,435930-015719-404591-282722-634139-131791-5761...,30DFFCC3-C1C0-4A30-8613-1AF352878257,505604-211002-055231-409310-028061-712019-2230...
2,LA220003,R2901304,E4-46-B0-18-39-BA,Intel(R) Ethernet Connection (16) I219-LM,イーサネット,3C-21-9C-CA-D8-5C,Intel(R) Wi-Fi 6E AX211 160MHz,Wi-Fi,AFDDD726-22F0-4C33-8564-DC1B3BE9E56A,462341-682275-561968-504845-278751-274021-2386...,E0D46914-9F3B-404C-8772-4A04FA445C45,377377-599335-384725-436656-221991-260612-5949...
3,LA220004,R2901280,E4-46-B0-18-39-88,Intel(R) Ethernet Connection (16) I219-LM,イーサネット,3C-21-9C-CA-E7-3E,Intel(R) Wi-Fi 6E AX211 160MHz,Wi-Fi,4FF0E8FF-F879-4C87-9F2E-0EB07DBEC98B,402721-664048-368566-555907-665445-384758-5422...,8BA10322-55B2-45CE-AB5D-87D53F605A4D,031636-292193-345917-661144-057574-014047-4624...
4,LA220005,R2901395,E4-46-B0-18-3F-40,Intel(R) Ethernet Connection (16) I219-LM,イーサネット,3C-21-9C-C9-63-FA,Intel(R) Wi-Fi 6E AX211 160MHz,Wi-Fi,54773D83-9FB1-4173-8E96-7FC37B6A9106,413787-388487-159863-323279-357203-152581-1556...,95B39820-4532-421B-87F4-B6426065F1EB,184756-447722-663729-318582-033165-642763-6913...
...,...,...,...,...,...,...,...,...,...,...,...,...
316,LA230092,R3900120,E4-46-B0-3F-B9-84,Intel(R) Ethernet Connection (16) I219-LM,イーサネット,E0-2E-0B-BC-A6-C9,Intel(R) Wi-Fi 6E AX211 160MHz,Wi-Fi,DE459696-BC09-4CD9-A713-3DBD351E1A85,606012-121561-669867-230340-362725-406846-5182...,40D90B1A-E69F-48FB-B0EE-774A4F350FA4,397430-179179-627770-231363-612216-542168-5033...
317,LA230093,R3900060,E4-46-B0-3F-B8-69,Intel(R) Ethernet Connection (16) I219-LM,イーサネット,E0-2E-0B-BB-30-55,Intel(R) Wi-Fi 6E AX211 160MHz,Wi-Fi,84F674F7-E35B-4A17-8D91-453F393E5A32,357214-693352-711821-314732-085206-265518-3210...,5E88C152-F152-4F0E-8B04-6A6BB458DB1D,100056-427306-180213-496507-273075-081037-1186...
318,LA230094,R3900080,E4-46-B0-3F-B8-C9,Intel(R) Ethernet Connection (16) I219-LM,イーサネット,E0-2E-0B-BA-69-95,Intel(R) Wi-Fi 6E AX211 160MHz,Wi-Fi,385D60B8-C834-444C-9AAA-2951A85E99D7,321057-309529-000682-460834-557436-417835-1790...,44DC2CA6-CA0D-40F8-8F22-5CA89D8E2BAC,211717-225467-652454-189926-058487-111232-4474...
319,LA230098,R3900073,E4-46-B0-3F-B8-A8,Intel(R) Ethernet Connection (16) I219-LM,イーサネット,E0-2E-0B-BB-30-3C,Intel(R) Wi-Fi 6E AX211 160MHz,Wi-Fi,8FF10FCB-955B-4342-8E93-3E35CB486049,472626-211255-364859-681868-596629-246763-6524...,93CF3B66-0AF8-490F-A421-89280499222E,307197-028160-585970-057563-347666-674531-5968...


In [6]:
# --- セル3: Excel出力・ダウンロード ---
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

OUTPUT = '/content/PC情報一覧.xlsx'

wb = Workbook()
ws = wb.active
ws.title = 'PC情報一覧'

headers = list(df.columns)
thin_border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin'))
body_font = Font(name='Arial', size=10)

# カテゴリ別の色
fill_base = PatternFill('solid', fgColor='4472C4')   # 青: 基本情報
fill_nic  = PatternFill('solid', fgColor='548235')   # 緑: NIC
fill_key  = PatternFill('solid', fgColor='BF8F00')   # 黄: 回復キー
header_font = Font(bold=True, color='FFFFFF', name='Arial', size=10)

for col, h in enumerate(headers, 1):
    cell = ws.cell(row=1, column=col, value=h)
    cell.font = header_font
    cell.alignment = Alignment(horizontal='center', wrap_text=True)
    cell.border = thin_border
    if h.startswith('NIC'):
        cell.fill = fill_nic
    elif 'ドライブ' in h:
        cell.fill = fill_key
    else:
        cell.fill = fill_base

for r_idx, row_data in df.iterrows():
    for c_idx, col_name in enumerate(headers):
        cell = ws.cell(row=r_idx+2, column=c_idx+1, value=row_data[col_name])
        cell.font = body_font
        cell.border = thin_border

# 列幅自動調整
for col_idx, col_name in enumerate(headers, 1):
    max_len = len(str(col_name))
    for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
        for cell in row:
            if cell.value:
                max_len = max(max_len, len(str(cell.value)))
    ws.column_dimensions[get_column_letter(col_idx)].width = min(max_len + 3, 55)

ws.auto_filter.ref = ws.dimensions
ws.freeze_panes = 'A2'

wb.save(OUTPUT)
print(f'出力完了: {OUTPUT}')
colab_files.download(OUTPUT)

出力完了: /content/PC情報一覧.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>